In [323]:
import pandas as pd

df_sch_info = pd.read_csv('General information of schools.csv')
df_subjects = pd.read_csv('Subjects Offered.csv')
df_distprogrammes = pd.read_csv('School Distinctive Programmes.csv')
df_ccas = pd.read_csv('Co-curricular activities (CCAs).csv')

In [324]:
import openpyxl

df_affiliations = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=0)
df_psle = pd.read_excel('school_affiliations_psle.xlsx', sheet_name=1)

print(df_affiliations)
print(df_psle.head())

                                     school_name  \
0                  Anglo-Chinese School (Junior)   
1                  Anglo-Chinese School (Junior)   
2                 Anglo-Chinese School (Primary)   
3                 Anglo-Chinese School (Primary)   
4                           Catholic High School   
5                             De La Salle School   
6                             De La Salle School   
7                       Maris Stella High School   
8                         Montfort Junior School   
9                     St. Andrew's Junior School   
10                  St. Anthony's Primary School   
11                  St. Anthony's Primary School   
12                  St. Gabriel's Primary School   
13               St. Joseph's Institution Junior   
14               St. Joseph's Institution Junior   
15                          St. Stephen's School   
16                          St. Stephen's School   
17                Canossa Convent Primary School   
18          

In [325]:
import requests
from bs4 import BeautifulSoup

url = "https://www.property2b2c.com/school-ranking/2025-pop"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

tables = pd.read_html(str(soup.find_all("table")))
df_ballot = tables[0]

print(df_ballot)



/var/folders/2s/sj7n8qbs5838w7lml1xsqlv40000gn/T/ipykernel_6120/2403609962.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup.find_all("table")))


    No.                School  Unnamed: 2  GEP  SAP Gender           Area  \
0     1               Tao Nan         NaN  GEP  SAP    NaN  Marine Parade   
1     2               Ai Tong         NaN  NaN  SAP    NaN         Bishan   
2     3               Nanyang         NaN  GEP  SAP    NaN    Bukit Timah   
3     4  Pei Hwa Presbyterian         NaN  NaN  SAP    NaN    Bukit Timah   
4     5      Methodist Girls'         NaN  NaN  NaN   Girl    Bukit Timah   
..   ..                   ...         ...  ...  ...    ...            ...   
174   -                Yishun         NaN  NaN  NaN    NaN         Yishun   
175   -                 Yumin         NaN  NaN  NaN    NaN       Tampines   
176   -               Zhangde         NaN  NaN  NaN    NaN    Bukit Merah   
177   -              Zhenghua         NaN  NaN  NaN    NaN  Bukit Panjang   
178   -              Zhonghua         NaN  NaN  NaN    NaN      Serangoon   

     Vacancy  Applicant Admission Popularity  
0         20         63     

In [326]:
df_sch_info = df_sch_info[df_sch_info['mainlevel_code'].isin(['PRIMARY', 'MIXED LEVEL (P1-S4)'])]
df_sch_info = df_sch_info[['school_name', 'address', 'postal_code', 'nature_code', 'sap_ind', 'autonomous_ind', 'gifted_ind']]
df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})
df_sch_info['school_name'] = df_sch_info['school_name'].replace(
    "ST ANDREW'S SCHOOL (JUNIOR)", 
    "ST. ANDREW'S JUNIOR SCHOOL"
)
print(df_sch_info)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
2    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
4                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
5        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
6     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
327          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
329          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
332        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
333       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
335       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

/var/folders/2s/sj7n8qbs5838w7lml1xsqlv40000gn/T/ipykernel_6120/1684637911.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sch_info = df_sch_info.replace({'na': 0, 'Yes': 1, 'No': 0})


In [327]:
from rapidfuzz import process, fuzz

def fuzzy_match_schools(school_name, sch_info_names, threshold=60):
    # Try token_set_ratio first
    match, score, _ = process.extractOne(
        school_name,
        sch_info_names,
        scorer=fuzz.token_set_ratio
    )
    if score >= threshold:
        return match
    
    # Fallback: strip common suffixes from ground truth and try again
    suffixes = ["PRIMARY SCHOOL", "SCHOOL", "PRIMARY"]
    
    stripped_names = {}
    for name in sch_info_names:
        stripped = name.upper()
        for suffix in suffixes:
            if stripped.endswith(suffix):
                stripped = stripped[:-len(suffix)].strip()
                break
        stripped_names[stripped] = name  # map stripped -> original
    
    match, score, _ = process.extractOne(
        school_name.upper(),
        list(stripped_names.keys()),
        scorer=fuzz.token_set_ratio
    )
    
    if score >= threshold:
        return stripped_names[match]  # return original full name
    
    return None

In [328]:
# Apply fuzzy matching to map df_ballot schools to df_sch_info
df_ballot['school_name_matched'] = df_ballot['School'].apply(
    lambda x: fuzzy_match_schools(x, df_sch_info['school_name'].values, threshold=60)
)

# Check for unmatched schools
unmatched = df_ballot[df_ballot['school_name_matched'].isna()]
if len(unmatched) > 0:
    print(f"Unmatched schools:\n{unmatched[['School', 'school_name_matched']]}")
else:
    print("All schools matched successfully")

print(df_ballot[['School', 'school_name_matched']].head(10))

All schools matched successfully
                     School                  school_name_matched
0                   Tao Nan                       TAO NAN SCHOOL
1                   Ai Tong                       AI TONG SCHOOL
2                   Nanyang               NANYANG PRIMARY SCHOOL
3      Pei Hwa Presbyterian  PEI HWA PRESBYTERIAN PRIMARY SCHOOL
4          Methodist Girls'    METHODIST GIRLS' SCHOOL (PRIMARY)
5                 Nan Chiau             NAN CHIAU PRIMARY SCHOOL
6  CHIJ St. Nicholas Girls'      CHIJ ST. NICHOLAS GIRLS' SCHOOL
7          CHIJ (Toa Payoh)             CHIJ PRIMARY (TOA PAYOH)
8              Red Swastika                  RED SWASTIKA SCHOOL
9                  Kong Hwa                      KONG HWA SCHOOL


In [329]:
df_ballot = df_ballot[['school_name_matched', 'Vacancy', 'Applicant', 'Admission', 'Popularity']]
df_ballot = df_ballot.rename(columns={
    'school_name_matched': 'school_name',
    'Vacancy': '2B_vacancies',
    'Applicant': '2B_applicants',
    'Admission': '2B_admission_rate',
    'Popularity': '2B_popularity_score' })
df_ballot['2B_admission_rate'] = df_ballot['2B_admission_rate'].str.rstrip('%').astype(float)
print(df_ballot)

                             school_name  2B_vacancies  2B_applicants  \
0                         TAO NAN SCHOOL            20             63   
1                         AI TONG SCHOOL            20             62   
2                 NANYANG PRIMARY SCHOOL            20             53   
3    PEI HWA PRESBYTERIAN PRIMARY SCHOOL            20             51   
4      METHODIST GIRLS' SCHOOL (PRIMARY)            22             54   
..                                   ...           ...            ...   
174                YISHUN PRIMARY SCHOOL            40              0   
175                 YUMIN PRIMARY SCHOOL            67              0   
176               ZHANGDE PRIMARY SCHOOL            29              0   
177              ZHENGHUA PRIMARY SCHOOL            20              0   
178              ZHONGHUA PRIMARY SCHOOL            44              0   

     2B_admission_rate 2B_popularity_score  
0                 31.7                3.15  
1                 32.3           

In [330]:
# Apply fuzzy matching to map df_distprogrammes schools to df_sch_info
df_distprogrammes['school_name_matched'] = df_distprogrammes['school_name'].apply(
    lambda x: fuzzy_match_schools(x, df_sch_info['school_name'].values, threshold=60)
)
df_distprogrammes = df_distprogrammes[['school_name_matched', 'alp_domain']]
df_distprogrammes = df_distprogrammes.rename(columns={'school_name_matched': 'school_name'})
print(df_distprogrammes)

                       school_name                     alp_domain
0         ADMIRALTY PRIMARY SCHOOL   STEM - Emerging Technologies
1     AHMAD IBRAHIM PRIMARY SCHOOL                     Humanities
2        ANG MO KIO PRIMARY SCHOOL          STEM - Sustainability
3    ANGLO-CHINESE SCHOOL (JUNIOR)                      Languages
4                             None                      Languages
..                             ...                            ...
279           YUHUA PRIMARY SCHOOL              Interdisciplinary
280           YUMIN PRIMARY SCHOOL  STEM - Game Design and Making
281         ZHANGDE PRIMARY SCHOOL          STEM - Sustainability
282        ZHENGHUA PRIMARY SCHOOL              Interdisciplinary
283        ZHONGHUA PRIMARY SCHOOL          STEM - Sustainability

[284 rows x 2 columns]


In [331]:
# Filter to include only schools from df_sch_info
df_subjects = df_subjects.rename(columns={'School_Name': 'school_name'}) 
df_subjects = df_subjects[df_subjects['school_name'].isin(df_sch_info['school_name'])]
df_ccas = df_ccas.rename(columns={'School_name': 'school_name'})
df_ccas = df_ccas[df_ccas['school_name'].isin(df_sch_info['school_name'])]
df_distprogrammes = df_distprogrammes[df_distprogrammes['school_name'].isin(df_sch_info['school_name'])]

print(df_ccas.head())
print(df_subjects.head())
print(df_distprogrammes.head())

                school_name school_section  \
0  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
1  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
2  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
3  ADMIRALTY PRIMARY SCHOOL        PRIMARY   
4  ADMIRALTY PRIMARY SCHOOL        PRIMARY   

                      cca_grouping_desc            cca_generic_name  \
0                        ART AND CRAFTS         CLUBS AND SOCIETIES   
1                         CHINESE DANCE  VISUAL AND PERFORMING ARTS   
2                                 CHOIR  VISUAL AND PERFORMING ARTS   
3                 DESIGN AND INNOVATION         CLUBS AND SOCIETIES   
4  ENGLISH LANGUAGE, DRAMA AND DEBATING         CLUBS AND SOCIETIES   

  cca_customized_name  
0                  na  
1                  na  
2                  na  
3                  na  
4                  na  
                school_name                 Subject_Desc
0  ADMIRALTY PRIMARY SCHOOL                          Art
1  ADMIRALTY PRIMARY SCHOOL           

In [332]:
# Keep only relevant columns for df_ccas
df_ccas = df_ccas[['school_name', 'cca_grouping_desc', 'cca_generic_name']]

print(df_ccas.head())

                school_name                     cca_grouping_desc  \
0  ADMIRALTY PRIMARY SCHOOL                        ART AND CRAFTS   
1  ADMIRALTY PRIMARY SCHOOL                         CHINESE DANCE   
2  ADMIRALTY PRIMARY SCHOOL                                 CHOIR   
3  ADMIRALTY PRIMARY SCHOOL                 DESIGN AND INNOVATION   
4  ADMIRALTY PRIMARY SCHOOL  ENGLISH LANGUAGE, DRAMA AND DEBATING   

             cca_generic_name  
0         CLUBS AND SOCIETIES  
1  VISUAL AND PERFORMING ARTS  
2  VISUAL AND PERFORMING ARTS  
3         CLUBS AND SOCIETIES  
4         CLUBS AND SOCIETIES  


In [333]:
# Pivot df_ccas to create a counter of each cca_generic_name per school
df_ccas_pivot = df_ccas.groupby(['school_name', 'cca_generic_name']).size().unstack(fill_value=0)
df_ccas_pivot = df_ccas_pivot.reset_index()
df_ccas_pivot.columns = ['school_name', 'cca_clubs', 'cca_others', 'cca_sports', 'cca_uniformed', 'cca_arts']
print(df_ccas_pivot.head())

                    school_name  cca_clubs  cca_others  cca_sports  \
0      ADMIRALTY PRIMARY SCHOOL          5           0           5   
1  AHMAD IBRAHIM PRIMARY SCHOOL          3           1           4   
2                AI TONG SCHOOL          4           0           6   
3      ALEXANDRA PRIMARY SCHOOL          6           0           7   
4   ANCHOR GREEN PRIMARY SCHOOL          4           0           4   

   cca_uniformed  cca_arts  
0              2         6  
1              1         4  
2              2         5  
3              1         5  
4              1         5  


In [334]:
# Categorize alp_domain into distprog categories
def categorize_alp(domain):
    if 'STEM' in domain.upper():
        return 'STEM'
    elif 'ICT' in domain.upper():
        return 'ICT'
    elif 'INTERDISCIPLINARY' in domain.upper():
        return 'Interdisciplinary'
    elif 'LANGUAGES' in domain.upper():
        return 'Languages'

df_distprogrammes['alp_category'] = df_distprogrammes['alp_domain'].apply(categorize_alp)

# Pivot to count each category per school
df_distprogrammes_pivot = df_distprogrammes.groupby(['school_name', 'alp_category']).size().unstack(fill_value=0)
df_distprogrammes_pivot = df_distprogrammes_pivot.reset_index()
df_distprogrammes_pivot = df_distprogrammes_pivot.rename(columns={
    'STEM': 'distprog_stem',
    'ICT': 'distprog_ict',
    'Interdisciplinary': 'distprog_interdisciplinary',
    'Languages': 'distprog_languages'
})

print(df_distprogrammes_pivot)

alp_category                   school_name  distprog_ict  \
0                 ADMIRALTY PRIMARY SCHOOL             0   
1             AHMAD IBRAHIM PRIMARY SCHOOL             0   
2                 ALEXANDRA PRIMARY SCHOOL             0   
3              ANCHOR GREEN PRIMARY SCHOOL             0   
4                  ANDERSON PRIMARY SCHOOL             0   
..                                     ...           ...   
160                   YUHUA PRIMARY SCHOOL             0   
161                   YUMIN PRIMARY SCHOOL             0   
162                 ZHANGDE PRIMARY SCHOOL             0   
163                ZHENGHUA PRIMARY SCHOOL             0   
164                ZHONGHUA PRIMARY SCHOOL             0   

alp_category  distprog_interdisciplinary  distprog_languages  distprog_stem  
0                                      0                   0              2  
1                                      0                   0              1  
2                                      0     

In [335]:
# Pivot df_subjects to create a counter of each subject per school
df_subjects_pivot = df_subjects.groupby('school_name').size().reset_index(name='subject_count')

print(df_subjects_pivot)

                      school_name  subject_count
0        ADMIRALTY PRIMARY SCHOOL             24
1    AHMAD IBRAHIM PRIMARY SCHOOL             21
2                  AI TONG SCHOOL             14
3        ALEXANDRA PRIMARY SCHOOL             22
4     ANCHOR GREEN PRIMARY SCHOOL             30
..                            ...            ...
177          YUHUA PRIMARY SCHOOL             23
178          YUMIN PRIMARY SCHOOL             23
179        ZHANGDE PRIMARY SCHOOL             29
180       ZHENGHUA PRIMARY SCHOOL             25
181       ZHONGHUA PRIMARY SCHOOL             23

[182 rows x 2 columns]


In [336]:
df_affiliations = df_affiliations.copy()
df_affiliations['school_name'] = df_affiliations['school_name'].str.upper()
# Pivot df_affiliations to create a counter of affiliations per school
df_affiliations_pivot = df_affiliations.groupby('school_name').size().reset_index(name='affiliation_count')

df_affiliations_pivot

,school_name,affiliation_count
0,ANGLO-CHINESE SCHOOL (JUNIOR),2
1,ANGLO-CHINESE SCHOOL (PRIMARY),2
2,CANOSSA CONVENT PRIMARY SCHOOL,1
3,CATHOLIC HIGH SCHOOL,1
4,CHIJ (KATONG) PRIMARY,1
5,CHIJ (KELLOCK),1
6,CHIJ OUR LADY OF GOOD COUNSEL,1
7,CHIJ OUR LADY OF THE NATIVITY,1
8,CHIJ OUR LADY QUEEN OF PEACE,1
9,CHIJ PRIMARY (TOA PAYOH),1


In [337]:
df_psle['school_name'] = df_psle['school_name'].str.upper()
print(df_psle)

   rank                              school_name median_psle_score_range
0     1            RAFFLES GIRLS' PRIMARY SCHOOL                     4-7
1     2                   NANYANG PRIMARY SCHOOL                     4-7
2     3                     CATHOLIC HIGH SCHOOL                     5-8
3     4           ANGLO-CHINESE SCHOOL (PRIMARY)                     5-8
4     5                HENRY PARK PRIMARY SCHOOL                     5-8
5     6                            ROSYTH SCHOOL                     5-8
6     7                           AI TONG SCHOOL                     6-9
7     8        METHODIST GIRLS' SCHOOL (PRIMARY)                     6-9
8     9  SINGAPORE CHINESE GIRLS' PRIMARY SCHOOL                     6-9
9    10          CHIJ ST. NICHOLAS GIRLS' SCHOOL                     6-9


In [338]:
# Merge all datasets with df_sch_info as the base
df_schools = df_sch_info.copy()

# Merge df_ballot
df_schools = df_schools.merge(df_ballot, left_on='school_name', right_on='school_name', how='left')

# Merge df_subjects_pivot
df_schools = df_schools.merge(df_subjects_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_distprogrammes_pivot
df_schools = df_schools.merge(df_distprogrammes_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_ccas_pivot
df_schools = df_schools.merge(df_ccas_pivot, left_on='school_name', right_on='school_name', how='left')

# Merge df_affiliations_pivot
df_schools = df_schools.merge(df_affiliations_pivot, left_on='school_name', right_on='school_name', how='left')

# Add top_psle_score column
df_schools['top_psle_score'] = df_schools['school_name'].isin(df_psle['school_name']).astype(int)

# if NaN replace with 0
df_schools = df_schools.fillna(0)

# remove duplicate school_name if any
df_schools = df_schools.drop_duplicates(subset=['school_name'])

print(df_schools)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
1    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
2                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
3        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
4     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
183          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
184          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
185        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
186       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
187       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

In [339]:
# Convert numeric columns to integer, excluding Popularity and Admission
numeric_cols = df_schools.select_dtypes(include=['float64', 'int64']).columns
cols_to_convert = [col for col in numeric_cols if col not in ['2B_popularity_score', '2B_admission_rate']]

for col in cols_to_convert:
    df_schools[col] = df_schools[col].astype('int64')

print(df_schools)

                      school_name                        address  postal_code  \
0        ADMIRALTY PRIMARY SCHOOL         11 WOODLANDS CIRCLE          738907   
1    AHMAD IBRAHIM PRIMARY SCHOOL         10 YISHUN STREET 11          768643   
2                  AI TONG SCHOOL       100 Bright Hill Drive          579646   
3        ALEXANDRA PRIMARY SCHOOL  2A Prince Charles Crescent          159016   
4     ANCHOR GREEN PRIMARY SCHOOL         31 Anchorvale Drive          544969   
..                            ...                            ...          ...   
183          YUHUA PRIMARY SCHOOL   158 JURONG EAST STREET 24          609558   
184          YUMIN PRIMARY SCHOOL        3 TAMPINES STREET 21          529393   
185        ZHANGDE PRIMARY SCHOOL            51 Jalan Membina          169485   
186       ZHENGHUA PRIMARY SCHOOL                9 Fajar Road          679002   
187       ZHONGHUA PRIMARY SCHOOL       12 SERANGOON AVENUE 4          556095   

      nature_code  sap_ind 

In [340]:
df_schools.to_csv('schools_data.csv', index=False)

In [341]:
df_schools

,school_name,address,postal_code,nature_code,sap_ind,autonomous_ind,gifted_ind,2B_vacancies,2B_applicants,2B_admission_rate,...,distprog_interdisciplinary,distprog_languages,distprog_stem,cca_clubs,cca_others,cca_sports,cca_uniformed,cca_arts,affiliation_count,top_psle_score
0,ADMIRALTY PRIMARY SCHOOL,11 WOODLANDS CIRCLE,738907,CO-ED SCHOOL,0,0,0,26,27,96.3,...,0,0,2,5,0,5,2,6,0,0
1,AHMAD IBRAHIM PRIMARY SCHOOL,10 YISHUN STREET 11,768643,CO-ED SCHOOL,0,0,0,62,1,100.0,...,0,0,1,3,1,4,1,4,0,0
2,AI TONG SCHOOL,100 Bright Hill Drive,579646,CO-ED SCHOOL,1,0,0,20,62,32.3,...,0,0,0,4,0,6,2,5,0,1
3,ALEXANDRA PRIMARY SCHOOL,2A Prince Charles Crescent,159016,CO-ED SCHOOL,0,0,0,23,5,100.0,...,0,0,1,6,0,7,1,5,0,0
4,ANCHOR GREEN PRIMARY SCHOOL,31 Anchorvale Drive,544969,CO-ED SCHOOL,0,0,0,52,0,100.0,...,1,0,0,4,0,4,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,YUHUA PRIMARY SCHOOL,158 JURONG EAST STREET 24,609558,CO-ED SCHOOL,0,0,0,58,1,100.0,...,1,0,1,4,0,7,1,6,0,0
184,YUMIN PRIMARY SCHOOL,3 TAMPINES STREET 21,529393,CO-ED SCHOOL,0,0,0,67,0,100.0,...,0,0,1,3,0,3,2,4,0,0
185,ZHANGDE PRIMARY SCHOOL,51 Jalan Membina,169485,CO-ED SCHOOL,0,0,0,29,0,100.0,...,0,0,1,1,0,8,2,2,0,0
186,ZHENGHUA PRIMARY SCHOOL,9 Fajar Road,679002,CO-ED SCHOOL,0,0,0,20,0,100.0,...,1,0,1,3,0,4,2,6,0,0
